# Pruebas de hipótesis

Este notebook continúa el análisis a partir del dataset limpio generado en `exploration.ipynb`.


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import levene

import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

df_clean = pd.read_csv('../data/spotify_tracks_clean.csv')
print(f"Filas cargadas: {len(df_clean)}")

## Estadística descriptiva

A continuación calculamos estadísticas descriptivas para las variables numéricas del dataset limpio.


In [ ]:
cols_interes = ['popularity', 'duration_ms', 'danceability', 'energy',
                    'key', 'loudness', 'mode', 'speechiness', 'acousticness', 
                    'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']

df_clean[cols_interes].describe()

### Sesgo en las columnas

A partir de la tabla anterior, algunas columnas muestran sesgo notable:

- **Alto sesgo:** `instrumentalness`, `speechiness`, `acousticness`, `liveness`
- **Resto de columnas:** distribución relativamente equilibrada

Para profundizar, generaremos histogramas de las columnas relevantes.


In [ ]:
cols_clave = ['popularity', 'duration_ms', 'danceability', 'energy', 'loudness', 
              'valence', 'tempo', 'speechiness', 'acousticness', 
              'instrumentalness', 'liveness', 'key']

df_clean[cols_clave].hist(figsize=(15, 12), bins=30, edgecolor='black')
plt.tight_layout()
plt.show()

### Interpretación de la distribución

**Columnas con mayor sesgo:** `popularity`, `duration_ms`, `acousticness`, `instrumentalness`, `speechiness` y `liveness` concentran muchos valores en rangos específicos y presentan colas largas. También hay sesgo moderado en `tempo` y `loudness`.

**Columnas más equilibradas:** `danceability`, `energy` y `valence` muestran una distribución más uniforme, lo que las hace más estables para comparar canciones.

**Variables discretas:** `key`, `mode` y `time_signature` se comportan como variables categóricas, no continuas.


In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='mode', y='valence', data=df_clean)
plt.xlabel('Mode (0 = menor, 1 = mayor)')
plt.ylabel('Valence')
plt.title('Valence según Mode')
plt.show()

## ¿Existe relación entre `mode` y `valence`?

Exploramos si la tonalidad de una canción (mayor o menor) se relaciona con su valencia emocional.

### Observación visual

El boxplot sugiere que **no hay una diferencia marcada** entre los grupos: los rangos intercuartílicos de `valence` son bastante similares para modo mayor y menor.

Aun así, contrastaremos las medias con pruebas estadísticas para distinguir variación aleatoria de un posible efecto de `mode`.

### Plan de pruebas

1. **Levene** — comprobar si las varianzas son iguales (supuesto del *t-test* de Student).
2. ***t*-test de Welch** — comparar medias cuando no conocemos la varianza poblacional o las varianzas difieren.
3. **Cohen's *d*** — medir si la diferencia, aunque significativa, tiene relevancia práctica.

> La librería `scipy.stats` facilita estos cálculos. Para más detalle sobre cada prueba, conviene revisar la teoría detrás de cada una.


### Comprobación de varianzas (test de Levene)

Antes del *t-test*, verificamos si las varianzas de ambos grupos son equivalentes.

Usamos **Levene** en lugar de un test basado en χ² porque no podemos asumir normalidad en las poblaciones. El p-value resultante nos indica si las varianzas pueden considerarse iguales.


In [ ]:
from scipy import stats
from scipy.stats import levene

mayor = df_clean[df_clean['mode'] == 1]['valence']
menor = df_clean[df_clean['mode'] == 0]['valence']

print(f"Valence promedio - Modo mayor: {mayor.mean():.4f}")
print(f"Valence promedio - Modo menor: {menor.mean():.4f}")
print(f"tamaño de muestra - Modo mayor: {len(mayor)}")
print(f"tamaño de muestra - Modo menor: {len(menor)}")

stat, p_levene = levene(mayor, menor)
print(f"Levene p-value: {p_levene:.5f}")

### Resultado de Levene

Con **p-value = 0,00973** (< 0,05), rechazamos la hipótesis de varianzas iguales.

**Implicación:** no aplicamos el *t-test* de Student; usamos el **t-test de Welch** (`equal_var=False`), válido cuando las varianzas difieren.


In [ ]:
#ttest es t-test welch puesto que equal_var=False
t_stat, p_value = stats.ttest_ind(mayor, menor, equal_var=False)
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.10f}")

### Resultado del t-test de Welch

El **p-value ≈ 0** indica que la diferencia entre medias es **estadísticamente significativa** (no parece deberse al azar).

> ⚠️ Con muestras muy grandes (~89.000 registros), un p-value bajo no implica necesariamente un efecto **relevante** en la práctica.

Por eso calculamos el **tamaño del efecto** con Cohen's *d*.


In [ ]:
import numpy as np

def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    #Varianza de cada grupo
    var1, var2 = group1.var(ddof=1), group2.var(ddof=1)
    #Desviación estandar combinada
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (group1.mean() - group2.mean()) / pooled_std

d = cohens_d(mayor, menor)
print(f"Cohen's d: {d:.4f}")

## Conclusión: ¿Hay relación entre `mode` y `valence`?

### Definición de las variables

| Variable | Descripción |
|----------|-------------|
| **`mode`** | Tonalidad: `0` = menor (suele sonar triste), `1` = mayor (suele sonar alegre) |
| **`valence`** | Estado de ánimo: valores altos = positivo/alegre, valores bajos = negativo/triste |

### Resumen del análisis

| Paso | Resultado |
|------|-----------|
| Boxplot | Distribuciones muy similares entre modos |
| Media modo mayor | 0,4759 |
| Media modo menor | 0,4619 |
| Diferencia absoluta | 0,014 (~1,4 puntos porcentuales) |
| *t*-test de Welch | p-value ≈ 0 → diferencia significativa |
| Cohen's *d* | **0,0532** → efecto **despreciable** (< 0,2) |

### Interpretación

La intuición musical sugiere que las canciones en **modo mayor** deberían tener mayor `valence`. Los datos muestran una tendencia mínima en esa dirección, pero:

1. **Visualmente**, las distribuciones casi se superponen.
2. **Estadísticamente**, la diferencia es significativa — sobre todo porque el tamaño muestral es enorme (~89.000 canciones).
3. **Prácticamente**, Cohen's *d* = 0,053 indica un efecto **despreciable**: la diferencia entre medias no llega ni al **20%** de una desviación estándar.

### Conclusión final

**No hay una relación aparente ni relevante entre `mode` y `valence` en este dataset.** Aunque el test rechaza la hipótesis nula, el tamaño del efecto es tan pequeño que, para fines analíticos o de modelado, `mode` no parece ser un predictor útil de la valencia emocional de una canción.
